# LHV LIK75 XIRR Simulation

Simulates buying **1000 EUR worth of LHV Pensionifond Indeks (LIK75)** on a chosen day of
each month (snapped to the closest available NAV price), then computes XIRR using the same
Newton-Raphson + Bisection algorithm as the production `Xirr.kt`.

Data source: <https://www.lhv.ee/b/public/market-data/fund/LIK75?timeSpan=all>

**Configurable** (see Configuration cell): `MONTHLY_AMOUNT`, `TARGET_DAY_OF_MONTH`, `START_DATE`
(set to `null` for since-inception).

**Requirements**: Kotlin Jupyter kernel (`pip install kotlin-jupyter-kernel` or via conda).

In [29]:
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-serialization-json-jvm:1.7.3")
@file:DependsOn("org.apache.commons:commons-math3:3.6.1")

In [30]:
import kotlinx.serialization.json.*
import org.apache.commons.math3.analysis.differentiation.DerivativeStructure
import org.apache.commons.math3.analysis.differentiation.UnivariateDifferentiableFunction
import org.apache.commons.math3.analysis.solvers.BisectionSolver
import org.apache.commons.math3.analysis.solvers.NewtonRaphsonSolver
import java.net.URL
import java.time.Instant
import java.time.LocalDate
import java.time.YearMonth
import java.time.ZoneOffset
import java.time.temporal.ChronoUnit
import kotlin.math.abs
import kotlin.math.pow

## Configuration

In [31]:
val FUND_URL = "https://www.lhv.ee/b/public/market-data/fund/LIK75?timeSpan=all"
val MONTHLY_AMOUNT = 1000.0
val TARGET_DAY_OF_MONTH = 15
val START_DATE: LocalDate? = LocalDate.of(2016, 11, 7)  // null = since fund inception

## Domain types

In [32]:
data class PricePoint(val date: LocalDate, val price: Double)
data class CashFlow(val amount: Double, val date: LocalDate)
data class Purchase(val month: YearMonth, val buyDate: LocalDate, val nav: Double, val units: Double)

  ## Fetch & parse LHV price history

In [33]:
val raw = URL(FUND_URL).openConnection().apply {
  setRequestProperty("User-Agent", "Mozilla/5.0 (KotlinNotebook)")
}.getInputStream().bufferedReader().use { it.readText() }

val root = Json.parseToJsonElement(raw).jsonObject

val prices = root["priceGraphDetails"]!!.jsonArray
  .map {
    val obj = it.jsonObject
    PricePoint(
      date = Instant.parse(obj["timestamp"]!!.jsonPrimitive.content).atZone(ZoneOffset.UTC).toLocalDate(),
      price = obj["price"]!!.jsonPrimitive.double,
    )
  }
  .sortedBy { it.date }

val fundData = root["fundData"]!!.jsonObject
val fundName = fundData["name"]!!.jsonPrimitive.content
val apiNav = fundData["nav"]!!.jsonPrimitive.double

println("Fund:          $fundName")
println("Price points:  ${prices.size}  (${prices.first().date} -> ${prices.last().date})")
println("Latest NAV:    $apiNav  (series last point: ${prices.last().price})")

Fund:          LHV Pensionifond Indeks
Price points:  696  (2016-11-07 -> 2026-05-14)
Latest NAV:    1.5248  (series last point: 1.52479)


## XIRR (mirror of `src/main/kotlin/.../xirr/Xirr.kt`)

In [34]:
class Xirr(private val cashFlows: List<CashFlow>) {
  private val endDate = cashFlows.maxOf { it.date }
  private val startDate = cashFlows.minOf { it.date }

  private fun yearsToEnd(date: LocalDate): Double =
    ChronoUnit.DAYS.between(date, endDate).toDouble() / 365.25

  private fun npv(rate: Double): Double =
    cashFlows.sumOf { it.amount * (1 + rate).pow(yearsToEnd(it.date)) }

  private fun npvDerivative(rate: Double): Double =
    cashFlows.sumOf { it.amount * yearsToEnd(it.date) * (1 + rate).pow(yearsToEnd(it.date) - 1) }

  private fun xirrFunction() = object : UnivariateDifferentiableFunction {
    override fun value(x: Double): Double = npv(x)
    override fun value(t: DerivativeStructure): DerivativeStructure =
      DerivativeStructure(t.freeParameters, t.order, npv(t.value), npvDerivative(t.value))
  }

  private fun initialGuess(): Double {
    val total = cashFlows.sumOf { it.amount }
    val deposits = cashFlows.filter { it.amount < 0 }.sumOf { -it.amount }
    val years = ChronoUnit.DAYS.between(startDate, endDate).toDouble() / 365.25
    if (years <= 0) return 0.0
    return ((total / deposits).pow(1 / years) - 1).coerceIn(-0.9, 0.9)
  }

  fun calculate(): Double = runCatching {
    NewtonRaphsonSolver().solve(1000, xirrFunction(), -0.99, 0.99, initialGuess())
  }.getOrElse {
    BisectionSolver().solve(1000, xirrFunction(), -0.99, 0.99)
  }
}

## Simulate monthly purchases & compute XIRR

In [36]:
fun closestPriceTo(target: LocalDate): PricePoint =
  prices.minBy { abs(ChronoUnit.DAYS.between(it.date, target)) }

val startMonth = START_DATE.let { YearMonth.from(it) } ?: YearMonth.from(prices.first().date)
val endMonth = YearMonth.from(prices.last().date)

val months = generateSequence(startMonth) { it.plusMonths(1) }
  .takeWhile { !it.isAfter(endMonth) }
  .toList()

val purchases = months.map { ym ->
  val pp = closestPriceTo(ym.atDay(TARGET_DAY_OF_MONTH))
  Purchase(ym, pp.date, pp.price, MONTHLY_AMOUNT / pp.price)
}

val totalUnits = purchases.sumOf { it.units }
val totalInvested = MONTHLY_AMOUNT * purchases.size
val finalDate = prices.last().date
val finalNav = prices.last().price
val finalValue = totalUnits * finalNav

val cashFlows = purchases.map { CashFlow(-MONTHLY_AMOUNT, it.buyDate) } + CashFlow(finalValue, finalDate)
val xirr = Xirr(cashFlows).calculate()
val absoluteReturn = finalValue / totalInvested - 1

println("=".repeat(54))
println("Period start:      ${purchases.first().buyDate}")
println("Period end:        $finalDate")
println("Months simulated:  ${purchases.size}")
println("Total invested:    %,12.2f EUR".format(totalInvested))
println("Total units held:  %,12.4f".format(totalUnits))
println("Final NAV:         %12.5f EUR".format(finalNav))
println("Final value:       %,12.2f EUR".format(finalValue))
println("Absolute gain:     %,12.2f EUR  (%.2f%%)".format(finalValue - totalInvested, absoluteReturn * 100))
println("XIRR (annualized): %12.4f %%".format(xirr * 100))
println("=".repeat(54))

Period start:      2016-11-17
Period end:        2026-05-14
Months simulated:  115
Total invested:      115,000.00 EUR
Total units held:  130,286.2436
Final NAV:              1.52479 EUR
Final value:         198,659.16 EUR
Absolute gain:        83,659.16 EUR  (72.75%)
XIRR (annualized):      11.2054 %


## Per-month detail

In [37]:
println("%-8s  %-12s  %10s  %10s  %12s".format("Month", "Buy date", "NAV", "Units", "Cum.units"))
println("-".repeat(60))
var cum = 0.0
purchases.forEach { p ->
  cum += p.units
  println("%-8s  %-12s  %10.5f  %10.4f  %12.4f".format(p.month, p.buyDate, p.nav, p.units, cum))
}

Month     Buy date             NAV       Units     Cum.units
------------------------------------------------------------
2016-11   2016-11-17       0.64930   1540.1201     1540.1201
2016-12   2016-12-17       0.67268   1486.5909     3026.7111
2017-01   2017-01-16       0.66174   1511.1675     4537.8786
2017-02   2017-02-15       0.68602   1457.6834     5995.5621
2017-03   2017-03-17       0.68322   1463.6574     7459.2194
2017-04   2017-04-16       0.69237   1444.3145     8903.5339
2017-05   2017-05-16       0.67879   1473.2097    10376.7436
2017-06   2017-06-15       0.69223   1444.6066    11821.3501
2017-07   2017-07-15       0.68249   1465.2229    13286.5731
2017-08   2017-08-14       0.67370   1484.3402    14770.9133
2017-09   2017-09-13       0.68206   1466.1467    16237.0600
2017-10   2017-10-13       0.70069   1427.1647    17664.2246
2017-11   2017-11-17       0.70965   1409.1454    19073.3700
2017-12   2017-12-17       0.71384   1400.8741    20474.2441
2018-01   2018-01-16    